In [ ]:
# Configuration and source registry.
# This notebook renders already computed outputs; it does not estimate models.

import os
import re
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

# Add conda DLL folders when the notebook kernel does not inherit the activated PATH.
CONDA_PREFIX = Path(sys.prefix)
DLL_DIRS = [
    CONDA_PREFIX,
    CONDA_PREFIX / 'Library' / 'mingw-w64' / 'bin',
    CONDA_PREFIX / 'Library' / 'usr' / 'bin',
    CONDA_PREFIX / 'Library' / 'bin',
    CONDA_PREFIX / 'Scripts',
]
_DLL_DIRECTORY_HANDLES = []
for dll_dir in DLL_DIRS:
    if dll_dir.exists():
        os.environ['PATH'] = str(dll_dir) + os.pathsep + os.environ.get('PATH', '')
        if hasattr(os, 'add_dll_directory'):
            _DLL_DIRECTORY_HANDLES.append(os.add_dll_directory(str(dll_dir)))
os.environ.setdefault('MPLBACKEND', 'Agg')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle

from docx import Document
from docx.enum.section import WD_ORIENT, WD_SECTION
from docx.enum.table import WD_CELL_VERTICAL_ALIGNMENT, WD_ROW_HEIGHT_RULE, WD_TABLE_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Cm, Pt


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'AGENTS.md').exists() and (candidate / 'docs' / 'PROJECT_MAP.md').exists():
            return candidate
    raise FileNotFoundError('Could not locate project root from AGENTS.md and docs/PROJECT_MAP.md')


PROJECT_ROOT = find_project_root()
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'thesis_tables'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DOCX_REQUESTED = OUTPUT_DIR / 'notebook12_all_tables.docx'
FIGURES_DIR = OUTPUT_DIR / 'figures_nb12'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def timestamp_variant(path):
    if not path.exists():
        return path
    candidate = path.with_name(f'{path.stem}_{RUN_TIMESTAMP}{path.suffix}')
    counter = 2
    while candidate.exists():
        candidate = path.with_name(f'{path.stem}_{RUN_TIMESTAMP}_{counter}{path.suffix}')
        counter += 1
    return candidate


OUTPUT_DOCX_PATH = timestamp_variant(OUTPUT_DOCX_REQUESTED)

# Observed category labels are fixed from the processed sales and ML panels.
OBSERVED_CATEGORY_VALUES = [
    'Cold Drinks', 'Dessert', 'Dinner', 'Ekskluderes',
    'Hot Drinks', 'Lunch', 'Salads', 'Sweets',
]

TRANSLATION = {
    'AvdelingID': 'Store ID',
    'AvdelingTekst': 'Store name',
    'Analyse_Kategori': 'Category',
    'Ukedag': 'Weekday',
    'Årstid': 'Season',
    'Helligdager': 'Holiday',
    'HelligdagNavn': 'Holiday name',
    'Antall_total': 'Units',
    'Antall_campaign': 'Campaign units',
    'Antall_ordinary': 'Ordinary units',
    'Avdeling_total': 'Store total',
    'DatoSolgt': 'Date',
    'NettoKr': 'Net revenue',
    'AntallArtikler': 'Items',
    'AntallSalg': 'Transactions',
    'Cold Drinks': 'Cold drinks',
    'Dessert': 'Dessert',
    'Dinner': 'Dinner',
    'Ekskluderes': 'Excluded',
    'Hot Drinks': 'Hot drinks',
    'Lunch': 'Lunch',
    'Salads': 'Salads',
    'Sweets': 'Sweets',
    'actual_meps_h0_h2': 'Actual MEPS, h0-h2',
    'synthetic_scenario_h3_h10': 'Synthetic scenario, h3-h10',
    'synthetic_conservative_climatology_drift': 'Synthetic conservative climatology drift',
    'deterministic_meps': 'Deterministic MEPS',
    'baseline_alpha_schedule': 'Baseline alpha schedule',
    'stronger_climatology_drift': 'Stronger climatology drift',
    'weaker_climatology_drift': 'Weaker climatology drift',
    'h0_10': 'h0-h10',
    'h0_2': 'h0-h2',
    'h3_10': 'h3-h10',
    'all_days': 'All days',
    'cold_tail': 'Cold tail',
    'hot_tail': 'Hot tail',
    'wet_tail': 'Wet tail',
    'dry_tail': 'Dry tail',
}

ROUNDING = {
    'percent_decimals': 1,
    'error_unit_decimals': 2,
    'unit_count_decimals': 0,
    'proportion_percent_decimals': 1,
    'regression_coefficient_decimals': 3,
    'regression_tstat_decimals': 2,
    'r_squared_decimals': 3,
    'p_value_decimals': 3,
    'generic_decimals': 2,
    'training_minutes_decimals': 1,
}

OUTPUT_STYLE = {
    'page_width_cm': 21.0,
    'page_height_cm': 29.7,
    'margin_cm': 2.5,
    'text_width_cm': 16.0,
    'landscape_text_width_cm': 24.7,
    'font_family': 'Times New Roman',
    'table_body_font_size_pt': 10.5,
    'caption_font_size_pt': 10.5,
    'note_font_size_pt': 10.0,
    'cell_padding_twips': 120,
    'booktabs_border_size': '8',
    'figure_width_cm': 16.0,
    'figure_dpi': 300,
    'figure_title_font_size_pt': 11,
    'figure_label_font_size_pt': 10,
    'figure_tick_font_size_pt': 9,
    'figure_legend_font_size_pt': 9,
}

HOUSE_STYLE = {
    'font_family': OUTPUT_STYLE['font_family'],
    'body_font_size_pt': OUTPUT_STYLE['table_body_font_size_pt'],
    'caption_font_size_pt': OUTPUT_STYLE['caption_font_size_pt'],
    'note_font_size_pt': OUTPUT_STYLE['note_font_size_pt'],
    'figure_width_cm': OUTPUT_STYLE['figure_width_cm'],
    'figure_dpi': OUTPUT_STYLE['figure_dpi'],
    'booktabs_border_size': OUTPUT_STYLE['booktabs_border_size'],
}

SOURCE_SPECS = {
    't01_weather_sources': {'path': 'outputs/thesis_tables/table_1_weather_sources.csv'},
    't02_weather_window': {'path': 'outputs/thesis_tables/Table_2_weather_window_specification_choice.csv'},
    't03_feature_registry': {'path': 'outputs/ml_panel/ml_panel_feature_registry.csv'},
    't04_model_family': {'path': 'outputs/ml_models/model_family_selection_summary.csv'},
    't05_regression_panel_a': {'path': 'outputs/regression_analysis/csv/TableX_booktabs1_panelA.csv'},
    't05_regression_panel_b': {'path': 'outputs/regression_analysis/csv/TableX_booktabs1_panelB.csv'},
    't06_category_weather': {'path': 'outputs/regression_analysis/csv/Table3_category_summary.csv'},
    't07_basket_focus': {'path': 'outputs/basket_analysis/focus_pairs.csv'},
    't08_alpha_schedule': {'path': 'outputs/lightgbm_rolling_origin/final_daily_m1_m2_m3/lightgbm_rolling_origin_alpha_schedule_final_daily_m1_m2_m3.csv'},
    't09_ml_store_day_hgroup': {'path': 'outputs/lightgbm_rolling_origin/final_daily_m1_m2_m3/results_report/tables/table_ml_store_day_metrics_by_horizon_group.csv'},
    't09_ml_overall_hgroup': {'path': 'outputs/lightgbm_rolling_origin/final_daily_m1_m2_m3/results_report/tables/table_ml_overall_metrics_by_horizon_group.csv'},
    't10_store_category_units': {'path': 'outputs/lightgbm_rolling_origin/final_daily_m1_m2_m3/results_report/tables/table_ml_store_category_day_unit_interpretation.csv'},
    't10_store_day_units': {'path': 'outputs/lightgbm_rolling_origin/final_daily_m1_m2_m3/results_report/tables/table_ml_store_day_metrics_by_horizon_group.csv'},
    't10_replenishment_units': {'path': 'outputs/lightgbm_rolling_origin/final_daily_m1_m2_m3/results_report/tables/table_ml_replenishment_h1_h8_m2_vs_m1_improvement.csv'},
    't11_category_improvement': {'path': 'outputs/lightgbm_rolling_origin/final_daily_m1_m2_m3/results_report/tables/table_ml_category_absolute_and_relative_improvement.csv'},
    't12_full_step5': {'path': 'outputs/thesis_tables/full_step5/Table_X_full_step5_main_results.csv'},
    't13_region_summary': {'path': 'outputs/lightgbm_rolling_origin/final_daily_m1_m2_m3/results_report/tables/table_ml_region_summary.csv'},
    't14_alpha_sensitivity': {'path': 'outputs/weather_emulator_alpha_sensitivity/m2_alpha_sensitivity/results_report/m2_alpha_vs_m1_overall_h3_h10_summary.csv'},
    't15_vif_panel_a': {'path': 'outputs/regression_analysis/csv/Table_A1_VIF_summary_panelA.csv'},
    't15_vif_panel_b': {'path': 'outputs/regression_analysis/csv/Table_A1_VIF_summary_panelB.csv'},
    't16_significance_panel_a': {'path': 'outputs/regression_analysis/csv/TableA2_significance_structure_panelA.csv'},
    't16_significance_panel_b': {'path': 'outputs/regression_analysis/csv/TableA2_significance_structure_panelB.csv'},
    't17_iqr_effects': {'path': 'outputs/regression_analysis/csv/TableA3_iqr_effects.csv'},
    't18_weather_shap': {'path': 'outputs/thesis_tables/interpretability/Table_X_m2_weather_feature_ranking.csv'},
    'fig_a_realised_weather': {
        'path': 'data/processed/realised_weather_daily_windows.parquet',
        'columns': ['date', 'AvdelingID', 'aggregation_window', 'temp_obs_mean', 'precip_obs_sum', 'wind_obs_mean', 'cloud_obs_mean'],
    },
    'fig_a_climatology': {
        'path': 'data/processed/era5_longrun_climatology_parameters.parquet',
        'columns': ['AvdelingID', 'month', 'aggregation_window', 'fallback_level', 'temp_clim_mean', 'precip_clim_mean', 'wind_clim_mean', 'cloud_clim_mean'],
    },
    'fig_b_daily_schedule': {'path': 'outputs/lightgbm_rolling_origin/m2_daily_origin/preflight_audit/origin_schedule_summary_full.csv'},
    'fig_b_step5_schedule': {'path': 'outputs/lightgbm_rolling_origin/preflight_audit/origin_schedule_summary_full.csv'},
    'fig_c_metrics_by_horizon': {'path': 'outputs/lightgbm_rolling_origin/final_daily_m1_m2_m3/results_report/tables/table_ml_metrics_by_horizon.csv'},
    'fig_d_m2_improvement': {'path': 'outputs/lightgbm_rolling_origin/final_daily_m1_m2_m3/results_report/tables/table_ml_m2_vs_m1_improvement_by_horizon.csv'},
    'fig_e_store_day_horizon': {'path': 'outputs/lightgbm_rolling_origin/final_daily_m1_m2_m3/results_report/tables/table_ml_store_day_metrics_by_horizon.csv'},
    'fig_f_replenishment_cycle': {'path': 'outputs/lightgbm_rolling_origin/final_daily_m1_m2_m3/results_report/tables/table_ml_replenishment_h1_h8_cycle_metrics.csv'},
    'fig_g_category_horizon': {'path': 'outputs/lightgbm_rolling_origin/final_daily_m1_m2_m3/results_report/tables/table_ml_category_horizon_summary.csv'},
    'fig_h_weather_tail': {'path': 'outputs/lightgbm_rolling_origin/final_daily_m1_m2_m3/results_report/tables/table_ml_weather_tail_summary.csv'},
    'fig_i_season': {'path': 'outputs/lightgbm_rolling_origin/final_daily_m1_m2_m3/results_report/tables/table_ml_season_summary.csv'},
    'fig_j_importance': {'path': 'outputs/thesis_tables/interpretability/Table_X_m2_grouped_importance.csv'},
    'fig_j_importance_raw': {'path': 'outputs/lightgbm_interpretability/tables/grouped_feature_importance_by_model.csv'},
    'fig_k_daily_vs_step5': {'path': 'outputs/lightgbm_rolling_origin/m2_daily_origin/results_report/m2_daily_vs_step5_by_horizon.csv'},
}

print(f'Project root: {PROJECT_ROOT}')
print(f'Word output path: {OUTPUT_DOCX_PATH.relative_to(PROJECT_ROOT)}')
print(f'Figure output directory: {FIGURES_DIR.relative_to(PROJECT_ROOT)}')


In [ ]:
# Load configured source files.
# Missing sources are recorded and skipped when referenced.

DATA = {}
SOURCE_STATUS = []


def read_source(key, spec):
    path = PROJECT_ROOT / spec['path']
    if not path.exists():
        SOURCE_STATUS.append(
            {'source_key': key, 'path': str(path), 'status': 'missing', 'rows': None, 'columns': None}
        )
        DATA[key] = None
        return
    try:
        if path.suffix.lower() == '.csv':
            df = pd.read_csv(path)
        elif path.suffix.lower() == '.parquet':
            df = pd.read_parquet(path, columns=spec.get('columns'))
        else:
            raise ValueError(f'Unsupported source type: {path.suffix}')
        DATA[key] = df
        SOURCE_STATUS.append(
            {'source_key': key, 'path': str(path), 'status': 'loaded', 'rows': len(df), 'columns': list(df.columns)}
        )
    except Exception as exc:
        DATA[key] = None
        SOURCE_STATUS.append(
            {'source_key': key, 'path': str(path), 'status': f'error: {exc}', 'rows': None, 'columns': None}
        )


for source_key, source_spec in SOURCE_SPECS.items():
    read_source(source_key, source_spec)

source_status_df = pd.DataFrame(SOURCE_STATUS)
print('Loaded sources:', int((source_status_df['status'] == 'loaded').sum()), '/', len(source_status_df))
missing_or_error = source_status_df.loc[source_status_df['status'] != 'loaded', ['source_key', 'path', 'status']]
if not missing_or_error.empty:
    print('Sources that will be skipped if referenced:')
    display(missing_or_error)
else:
    print('All configured sources loaded.')


In [ ]:
# Shared Word-table and matplotlib rendering helpers.

MANIFEST = {
    'tables': [],
    'figures': [],
    'figure_sources': [],
    'narrowed_tables': [],
    'fit_flags': [],
    'skipped': [],
}

DOC = Document()
CURRENT_SECTION_ORIENTATION = 'portrait'


def configure_section(section, orientation):
    section.top_margin = Cm(OUTPUT_STYLE['margin_cm'])
    section.bottom_margin = Cm(OUTPUT_STYLE['margin_cm'])
    section.left_margin = Cm(OUTPUT_STYLE['margin_cm'])
    section.right_margin = Cm(OUTPUT_STYLE['margin_cm'])
    if orientation == 'landscape':
        section.orientation = WD_ORIENT.LANDSCAPE
        section.page_width = Cm(OUTPUT_STYLE['page_height_cm'])
        section.page_height = Cm(OUTPUT_STYLE['page_width_cm'])
    else:
        section.orientation = WD_ORIENT.PORTRAIT
        section.page_width = Cm(OUTPUT_STYLE['page_width_cm'])
        section.page_height = Cm(OUTPUT_STYLE['page_height_cm'])


section = DOC.sections[0]
configure_section(section, 'portrait')
styles = DOC.styles
styles['Normal'].font.name = HOUSE_STYLE['font_family']
styles['Normal']._element.rPr.rFonts.set(qn('w:eastAsia'), HOUSE_STYLE['font_family'])
styles['Normal'].font.size = Pt(HOUSE_STYLE['body_font_size_pt'])
DOC.add_paragraph('Notebook 12 thesis tables').runs[0].bold = True
DOC.add_paragraph(
    'Presentation-only tables rendered from already-computed source files. '
    'Metrics and model estimates are not recomputed in this notebook.'
)

MODEL_PALETTE = {'M1': '#4D4D4D', 'M2': '#2F7E7E', 'M3': '#8FA1AC', 'M4': '#6B7B8C'}
SINGLE_SERIES_COLOR = MODEL_PALETTE['M2']
COMPARISON_GREY = MODEL_PALETTE['M1']
HEATMAP_CMAP = LinearSegmentedColormap.from_list('nb12_sand_teal', ['#F2E8D5', '#2F5E5E'])
FIGURE_STYLE_NAME = 'nb12_shared_serif_sand_teal'
CURRENT_SECTION_ORIENTATION = 'portrait'

plt.rcParams.update({
    'font.family': HOUSE_STYLE['font_family'],
    'font.size': 10,
    'axes.titlesize': OUTPUT_STYLE['figure_title_font_size_pt'],
    'axes.labelsize': OUTPUT_STYLE['figure_label_font_size_pt'],
    'xtick.labelsize': OUTPUT_STYLE['figure_tick_font_size_pt'],
    'ytick.labelsize': OUTPUT_STYLE['figure_tick_font_size_pt'],
    'legend.fontsize': OUTPUT_STYLE['figure_legend_font_size_pt'],
    'figure.dpi': HOUSE_STYLE['figure_dpi'],
    'savefig.dpi': HOUSE_STYLE['figure_dpi'],
    'savefig.facecolor': 'white',
    'figure.facecolor': 'white',
})


def set_document_orientation(orientation, new_page=False):
    global CURRENT_SECTION_ORIENTATION
    if new_page or CURRENT_SECTION_ORIENTATION != orientation:
        target = DOC.add_section(WD_SECTION.NEW_PAGE)
    else:
        target = DOC.sections[-1]
    configure_section(target, orientation)
    CURRENT_SECTION_ORIENTATION = orientation


def skip_item(item_id, reason):
    print(f'SKIPPED - {item_id}: {reason}')
    MANIFEST['skipped'].append({'item': item_id, 'reason': reason})


def require_sources(item_id, source_keys):
    missing = []
    for key in source_keys:
        if key not in DATA or DATA[key] is None:
            spec = SOURCE_SPECS.get(key, {})
            missing.append(f'{key} ({spec.get("path", "unknown path")})')
    if missing:
        skip_item(item_id, 'missing source(s): ' + '; '.join(missing))
        return False
    return True


def translate_value(value):
    if pd.isna(value):
        return ''
    text = str(value)
    return TRANSLATION.get(text, text)


def to_number(value):
    if value is None or pd.isna(value):
        return None
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)
    text = str(value).strip().replace(',', '').replace('%', '')
    if text in {'', 'n.a.', 'na', 'NA', 'nan'}:
        return None
    try:
        return float(text)
    except ValueError:
        return None


def format_p_value(value):
    num = to_number(value)
    if num is None:
        return translate_value(value)
    if num < 0.001:
        return '<0.001'
    return f'{num:.{ROUNDING["p_value_decimals"]}f}'


def format_regression_value(value):
    if value is None or pd.isna(value):
        return ''
    text = str(value).strip()
    if text == '' or text.lower() == 'nan':
        return ''
    match = re.match(r'^([+-]?\d+(?:\.\d+)?)(\*{1,3})?(?:\s*\(([+-]?\d+(?:\.\d+)?)\))?$', text)
    if not match:
        return text
    coef, stars, tstat = match.groups()
    if not stars and tstat is None:
        return text
    coef_text = f'{float(coef):.{ROUNDING["regression_coefficient_decimals"]}f}'
    stars = stars or ''
    if tstat is None:
        return f'{coef_text}{stars}'
    t_text = f'{float(tstat):.{ROUNDING["regression_tstat_decimals"]}f}'
    return f'{coef_text}{stars} ({t_text})'


def format_value(value, kind):
    if value is None or pd.isna(value):
        return ''
    if kind in {None, 'text'}:
        return translate_value(value)
    if kind == 'raw':
        return str(value)
    if kind == 'count':
        num = to_number(value)
        return translate_value(value) if num is None else f'{num:,.0f}'
    if kind in {'units', 'error_units'}:
        num = to_number(value)
        return translate_value(value) if num is None else f'{num:.{ROUNDING["error_unit_decimals"]}f}'
    if kind == 'number_1':
        num = to_number(value)
        return translate_value(value) if num is None else f'{num:.1f}'
    if kind == 'number_2':
        num = to_number(value)
        return translate_value(value) if num is None else f'{num:.2f}'
    if kind == 'number_3':
        num = to_number(value)
        return translate_value(value) if num is None else f'{num:.3f}'
    if kind == 'pct':
        num = to_number(value)
        return translate_value(value) if num is None else f'{num:.{ROUNDING["percent_decimals"]}f}%'
    if kind == 'proportion_pct':
        num = to_number(value)
        return translate_value(value) if num is None else f'{100 * num:.{ROUNDING["proportion_percent_decimals"]}f}%'
    if kind == 'pp':
        num = to_number(value)
        return translate_value(value) if num is None else f'{num:.{ROUNDING["percent_decimals"]}f}'
    if kind == 'r2':
        num = to_number(value)
        return translate_value(value) if num is None else f'{num:.{ROUNDING["r_squared_decimals"]}f}'
    if kind == 'minutes':
        num = to_number(value)
        return translate_value(value) if num is None else f'{num:.{ROUNDING["training_minutes_decimals"]}f}'
    if kind == 'pvalue':
        return format_p_value(value)
    if kind == 'regression':
        return format_regression_value(value)
    return translate_value(value)


def format_display_df(df, config):
    display_df = df.copy()
    keep = config.get('keep')
    if keep is not None:
        missing = [col for col in keep if col not in display_df.columns]
        if missing:
            raise KeyError('Missing configured columns: ' + ', '.join(missing))
        display_df = display_df.loc[:, keep]
    if config.get('sort_by'):
        display_df = display_df.sort_values(config['sort_by']).reset_index(drop=True)
    display_df = display_df.rename(columns=config.get('rename', {}))
    display_df = display_df.rename(columns=lambda col: TRANSLATION.get(col, col))
    for column, replacement in config.get('fill_values', {}).items():
        if column in display_df.columns:
            display_df[column] = display_df[column].where(display_df[column].notna(), replacement)
    formats = config.get('formats', {})
    default_format = config.get('default_format', 'text')
    for column in display_df.columns:
        kind = formats.get(column, default_format)
        display_df[column] = display_df[column].map(lambda value, k=kind: format_value(value, k))
    return display_df



def set_cell_margins(cell, top=None, start=None, bottom=None, end=None):
    margins = {'top': top, 'start': start, 'bottom': bottom, 'end': end}
    tc_pr = cell._tc.get_or_add_tcPr()
    tc_mar = tc_pr.first_child_found_in('w:tcMar')
    if tc_mar is None:
        tc_mar = OxmlElement('w:tcMar')
        tc_pr.append(tc_mar)
    for edge, value in margins.items():
        if value is None:
            continue
        element = tc_mar.find(qn(f'w:{edge}'))
        if element is None:
            element = OxmlElement(f'w:{edge}')
            tc_mar.append(element)
        element.set(qn('w:w'), str(value))
        element.set(qn('w:type'), 'dxa')


def set_cell_width(cell, width_cm):
    tc_pr = cell._tc.get_or_add_tcPr()
    tc_w = tc_pr.first_child_found_in('w:tcW')
    if tc_w is None:
        tc_w = OxmlElement('w:tcW')
        tc_pr.append(tc_w)
    tc_w.set(qn('w:w'), str(int(width_cm * 567)))
    tc_w.set(qn('w:type'), 'dxa')


def repeat_table_header(row):
    tr_pr = row._tr.get_or_add_trPr()
    tbl_header = tr_pr.find(qn('w:tblHeader'))
    if tbl_header is None:
        tbl_header = OxmlElement('w:tblHeader')
        tr_pr.append(tbl_header)
    tbl_header.set(qn('w:val'), 'true')


def is_compact_numeric_column(column_name):
    lower = str(column_name).lower()
    compact_tokens = [
        'rmse', 'mae', 'wape', 'bias', 'r2', 'r²', 'vif', 'n ', ' n', 'rows',
        'stores', 'horizon', 'model', '%', 'share', 'coverage', 'lift',
        'support', 'probability', 'p(', 'units', 'mean actual',
    ]
    return any(token in lower for token in compact_tokens)


def table_text_width_cm(orientation=None):
    orientation = orientation or CURRENT_SECTION_ORIENTATION
    return OUTPUT_STYLE['landscape_text_width_cm'] if orientation == 'landscape' else OUTPUT_STYLE['text_width_cm']


def allocate_column_widths(display_df, orientation=None, explicit_widths=None):
    if explicit_widths:
        return [explicit_widths.get(col) for col in display_df.columns]
    text_width = table_text_width_cm(orientation)
    weights = []
    for col in display_df.columns:
        series = display_df[col].astype(str)
        max_len = max([len(str(col)), *(series.map(len).head(200).tolist() if len(series) else [0])])
        if is_compact_numeric_column(col):
            weight = max(0.75, min(1.35, max_len / 12))
        elif max_len >= 45:
            weight = 3.2
        elif max_len >= 28:
            weight = 2.3
        else:
            weight = 1.6
        weights.append(weight)
    total = sum(weights) or 1.0
    widths = [max(0.8, text_width * weight / total) for weight in weights]
    scale = text_width / sum(widths)
    return [width * scale for width in widths]


def should_flag_portrait_fit(item_id, display_df, config):
    if config.get('page_orientation') == 'landscape':
        return True
    if config.get('table_role') == 'appendix':
        return False
    text_width = OUTPUT_STYLE['text_width_cm']
    pressure = 0.0
    for col in display_df.columns:
        sample = display_df[col].astype(str).head(200)
        max_len = max([len(str(col)), *(sample.map(len).tolist() if len(sample) else [0])])
        pressure += 1.0 if is_compact_numeric_column(col) else min(3.0, max_len / 18)
    return pressure > text_width / 1.55


def clear_cell_borders(cell):
    tc_pr = cell._tc.get_or_add_tcPr()
    borders = tc_pr.find(qn('w:tcBorders'))
    if borders is None:
        borders = OxmlElement('w:tcBorders')
        tc_pr.append(borders)
    for edge in ['top', 'left', 'bottom', 'right', 'insideH', 'insideV']:
        element = borders.find(qn(f'w:{edge}'))
        if element is None:
            element = OxmlElement(f'w:{edge}')
            borders.append(element)
        element.set(qn('w:val'), 'nil')


def set_cell_rule(cell, edge):
    tc_pr = cell._tc.get_or_add_tcPr()
    borders = tc_pr.find(qn('w:tcBorders'))
    if borders is None:
        borders = OxmlElement('w:tcBorders')
        tc_pr.append(borders)
    element = borders.find(qn(f'w:{edge}'))
    if element is None:
        element = OxmlElement(f'w:{edge}')
        borders.append(element)
    element.set(qn('w:val'), 'single')
    element.set(qn('w:sz'), HOUSE_STYLE['booktabs_border_size'])
    element.set(qn('w:space'), '0')
    element.set(qn('w:color'), '000000')


def set_cell_text(cell, text, bold=False, font_size=None, alignment=WD_ALIGN_PARAGRAPH.LEFT):
    cell.text = ''
    paragraph = cell.paragraphs[0]
    paragraph.alignment = alignment
    paragraph.paragraph_format.space_before = Pt(0)
    paragraph.paragraph_format.space_after = Pt(0)
    run = paragraph.add_run(str(text))
    run.bold = bold
    run.font.name = HOUSE_STYLE['font_family']
    run._element.rPr.rFonts.set(qn('w:eastAsia'), HOUSE_STYLE['font_family'])
    run.font.size = Pt(font_size or HOUSE_STYLE['body_font_size_pt'])
    cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER
    pad = OUTPUT_STYLE['cell_padding_twips']
    set_cell_margins(cell, top=pad, start=pad, bottom=pad, end=pad)

def apply_booktabs(table):
    row_count = len(table.rows)
    repeat_table_header(table.rows[0])
    for row_index, row in enumerate(table.rows):
        row.height_rule = WD_ROW_HEIGHT_RULE.AUTO
        for cell in row.cells:
            clear_cell_borders(cell)
            if row_index == 0:
                set_cell_rule(cell, 'top')
                set_cell_rule(cell, 'bottom')
            if row_index == row_count - 1:
                set_cell_rule(cell, 'bottom')

def render_table_df(item_id, df, config):
    page_orientation = config.get('page_orientation')
    if page_orientation:
        set_document_orientation(page_orientation, new_page=config.get('own_page', False))
    try:
        display_df = format_display_df(df, config)
    except Exception as exc:
        skip_item(item_id, f'table preparation failed: {exc}')
        return
    caption = config.get('caption', item_id)
    note = config.get('note', '')
    role = config.get('table_role') or ('appendix' if caption.lower().startswith('appendix') else 'main')
    caption_paragraph = DOC.add_paragraph()
    caption_run = caption_paragraph.add_run(caption)
    caption_run.bold = True
    caption_run.italic = role == 'appendix'
    caption_run.font.name = HOUSE_STYLE['font_family']
    caption_run.font.size = Pt(HOUSE_STYLE['caption_font_size_pt'])
    table = DOC.add_table(rows=1, cols=len(display_df.columns))
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.autofit = False
    widths = allocate_column_widths(
        display_df,
        orientation=page_orientation,
        explicit_widths=config.get('column_widths_cm'),
    )
    alignments = config.get('alignments', {})
    for j, column in enumerate(display_df.columns):
        alignment = alignments.get(
            column,
            WD_ALIGN_PARAGRAPH.CENTER if is_compact_numeric_column(column) else WD_ALIGN_PARAGRAPH.LEFT,
        )
        set_cell_width(table.rows[0].cells[j], widths[j])
        set_cell_text(table.rows[0].cells[j], column, bold=True, alignment=alignment)
    for _, row in display_df.iterrows():
        cells = table.add_row().cells
        for j, value in enumerate(row.tolist()):
            column = display_df.columns[j]
            alignment = alignments.get(
                column,
                WD_ALIGN_PARAGRAPH.CENTER if is_compact_numeric_column(column) else WD_ALIGN_PARAGRAPH.LEFT,
            )
            set_cell_width(cells[j], widths[j])
            set_cell_text(cells[j], value, alignment=alignment)
    apply_booktabs(table)
    if note:
        note_paragraph = DOC.add_paragraph()
        note_run = note_paragraph.add_run('Note. ' + note)
        note_run.italic = True
        note_run.font.name = HOUSE_STYLE['font_family']
        note_run.font.size = Pt(HOUSE_STYLE['note_font_size_pt'])
    DOC.add_paragraph('')
    table_manifest = {
        'item': item_id,
        'caption': caption,
        'rows': len(display_df),
        'columns': list(display_df.columns),
        'role': role,
        'orientation': page_orientation or CURRENT_SECTION_ORIENTATION,
    }
    MANIFEST['tables'].append(table_manifest)
    if config.get('narrowed'):
        MANIFEST['narrowed_tables'].append(table_manifest)
    if should_flag_portrait_fit(item_id, display_df, {**config, 'table_role': role}):
        MANIFEST['fit_flags'].append(
            {
                'item': item_id,
                'columns': len(display_df.columns),
                'reason': (
                    'Does not fit portrait A4 at 10-12 pt with 2.5 cm margins without a '
                    'main-text-vs-appendix decision.'
                ),
            }
        )
    print(f'WROTE TABLE - {item_id}: {len(display_df)} rows, {len(display_df.columns)} columns')
    if config.get('return_to_portrait'):
        set_document_orientation('portrait', new_page=True)

def render_table_from_source(item_id, source_key, config):
    if require_sources(item_id, [source_key]):
        render_table_df(item_id, DATA[source_key], config)


def figure_output_paths(stem):
    pdf_path = FIGURES_DIR / f'{stem}.pdf'
    png_path = FIGURES_DIR / f'{stem}.png'
    if pdf_path.exists() or png_path.exists():
        stem = f'{stem}_{RUN_TIMESTAMP}'
        pdf_path = FIGURES_DIR / f'{stem}.pdf'
        png_path = FIGURES_DIR / f'{stem}.png'
        counter = 2
        while pdf_path.exists() or png_path.exists():
            stem_variant = f'{stem}_{counter}'
            pdf_path = FIGURES_DIR / f'{stem_variant}.pdf'
            png_path = FIGURES_DIR / f'{stem_variant}.png'
            counter += 1
    return pdf_path, png_path


def log_figure_source(item_id, source_keys, plotted_df, first_rows=5):
    paths = []
    print(f'SOURCE CHECK - {item_id}')
    for key in source_keys:
        rel_path = SOURCE_SPECS[key]['path']
        paths.append(rel_path)
        print(f' - {rel_path}')
    preview = plotted_df.head(first_rows)
    if not preview.empty:
        print('First plotted rows:')
        print(preview.to_string(index=False))
    MANIFEST['figure_sources'].append({'item': item_id, 'sources': paths})


def style_figure(ax, grid_axis='y'):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('0.25')
    ax.spines['bottom'].set_color('0.25')
    ax.tick_params(colors='0.20')
    ax.margins(x=0.04, y=0.10)
    ax.set_axisbelow(True)  # gridlines behind bars, not over them
    ax.grid(False)
    if grid_axis:
        ax.grid(axis=grid_axis, color='0.90', linewidth=0.7)


def save_figure(item_id, stem, fig, note=None):
    pdf_path, png_path = figure_output_paths(stem)
    fig.tight_layout(pad=1.0)
    fig.savefig(pdf_path, bbox_inches='tight', dpi=HOUSE_STYLE['figure_dpi'])
    fig.savefig(png_path, bbox_inches='tight', dpi=HOUSE_STYLE['figure_dpi'])
    MANIFEST['figures'].append(
        {
            'item': item_id,
            'pdf': str(pdf_path),
            'png': str(png_path),
            'note': note or '',
            'style': FIGURE_STYLE_NAME,
        }
    )
    print(f'WROTE FIGURE - {item_id}: {pdf_path.relative_to(PROJECT_ROOT)} and {png_path.relative_to(PROJECT_ROOT)}')
    plt.close(fig)


def figure_size(height_ratio=0.62):
    width = HOUSE_STYLE['figure_width_cm'] / 2.54
    return (width, width * height_ratio)


def prettify_axis(ax):
    style_figure(ax, grid_axis='y')


def parse_percent_text_to_r2(value):
    if value is None or pd.isna(value):
        return value
    text = str(value).strip()
    if not text.endswith('%'):
        return value
    num = to_number(text)
    if num is None:
        return value
    return f'{num / 100:.{ROUNDING["r_squared_decimals"]}f}'


## Data chapter tables and figures

This section renders presentation-ready data-source and weather-window outputs from already computed source files.


In [ ]:
# Render Table 1: weather data sources.

config = {
    'keep': ['Weather source', 'Type', 'Role in the project', 'Use in ML feature sets'],
    'caption': 'Table 1. Weather data sources and their role',
    'note': 'Presentation table read from the existing thesis table source. It is qualitative metadata, not a model result.',
}
render_table_from_source('Table 1 - weather data sources', 't01_weather_sources', config)


In [ ]:
# Render Table 2: weather-window diagnostics.

config = {
    'keep': ['Window', 'Demand-unit share', 'h=2 mean coverage', 'h=2 precip. RMSE', 'Delta Adj. R2 (pp)', 'Recommended role'],
    'rename': {'Delta Adj. R2 (pp)': 'Delta adjusted R2 (pp)'},
    'formats': {'Demand-unit share': 'pct', 'h=2 mean coverage': 'pct', 'h=2 precip. RMSE': 'error_units', 'Delta adjusted R2 (pp)': 'pp'},
    'caption': 'Table 2. Weather-window diagnostics and specification choice',
    'note': 'Main-text version. The source table also contains order share, h=2 full coverage, h=2 precipitation MAE, h=2 precipitation miss rate, RMSE log, and MAE log; these are omitted here for width. Percent columns are shown on a 0-100 scale.',
    'narrowed': True,
}
render_table_from_source('Table 2 - weather-window diagnostics', 't02_weather_window', config)


In [ ]:
# Register deferred Figure A: weather drift plot.

item_id = 'Figure A - weather drift plot'
skip_item(
    item_id,
    'Deferred until an upstream notebook writes a compact drift-source CSV. Notebook 12 does not read or plot from the heavy final weather panel.',
)
print('Figure A deferred: create a compact drift-source CSV upstream before rendering this figure in notebook 12.')


## Methodology tables and figures

This section renders the feature-set, model-screening, synthetic-weather and origin-schedule outputs used to document the modelling design.


In [ ]:
# Render Table 3: ML feature-set design.

item_id = 'Table 3 - ML feature-set design'
if require_sources(item_id, ['t03_feature_registry']):
    registry = DATA['t03_feature_registry']
    flags = ['allowed_m1_baseline', 'allowed_m2_synthetic_point_weather', 'allowed_m3_perfect_information', 'allowed_m4_synthetic_engineered_weather']
    required = ['feature_group', *flags]
    if any(col not in registry.columns for col in required):
        skip_item(item_id, 'feature registry missing required design columns')
    else:
        design = registry.loc[registry['role'].eq('feature'), required].copy()
        design = design.groupby('feature_group', observed=True)[flags].any().reset_index()
        for flag in flags:
            design[flag] = design[flag].map({True: 'Yes', False: 'No'})
        config = {
            'keep': required,
            'rename': {'feature_group': 'Feature group', 'allowed_m1_baseline': 'M1 baseline', 'allowed_m2_synthetic_point_weather': 'M2 forecast weather', 'allowed_m3_perfect_information': 'M3 realised weather', 'allowed_m4_synthetic_engineered_weather': 'M4 engineered weather'},
            'caption': 'Table 3. ML feature-set design',
            'note': 'Presentation-only roll-up of the notebook 06 feature registry. Yes indicates that at least one feature in the group is allowed in the feature set; no metrics are recomputed.',
        }
        render_table_df(item_id, design, config)


In [ ]:
# Render Table 4: model-family screening.

main_config = {
    'keep': ['model_family', 'mean_rmse_M1', 'mean_rmse_M2', 'M2_vs_M1_rmse_improvement_pct', 'training_time_total_minutes', 'methodological_role'],
    'rename': {'model_family': 'Model family', 'mean_rmse_M1': 'RMSE M1', 'mean_rmse_M2': 'RMSE M2', 'M2_vs_M1_rmse_improvement_pct': 'M2 over M1 RMSE improvement', 'training_time_total_minutes': 'Training time (min)', 'methodological_role': 'Methodological role'},
    'formats': {'RMSE M1': 'error_units', 'RMSE M2': 'error_units', 'M2 over M1 RMSE improvement': 'pct', 'Training time (min)': 'minutes'},
    'caption': 'Table 4. Model-family selection summary',
    'note': 'Model-family SCREENING step from notebook 07, not the final rolling-origin evaluation. RMSE is in units; improvement is relative to M1.',
}
render_table_from_source('Table 4 - model-family screening main', 't04_model_family', main_config)

full_config = {
    'keep': list(DATA['t04_model_family'].columns) if DATA.get('t04_model_family') is not None else None,
    'rename': {'model_family': 'Model family', 'benchmark_status': 'Status', 'completed_configurations': 'Completed configs', 'failed_configurations': 'Failed configs', 'mean_rmse_M1': 'RMSE M1', 'mean_rmse_M2': 'RMSE M2', 'mean_rmse_M3': 'RMSE M3', 'mean_rmse_M4': 'RMSE M4', 'best_operational_feature_set_by_rmse': 'Best operational set', 'mean_mae_best_operational': 'MAE best operational', 'mean_wape_best_operational': 'WAPE best operational', 'M2_vs_M1_rmse_improvement_pct': 'M2 over M1 RMSE improvement', 'M4_vs_M2_rmse_improvement_pct': 'M4 over M2 RMSE improvement', 'actual_meps_h0_h2_M2_vs_M1_rmse_improvement_pct': 'M2 over M1, h0-h2', 'synthetic_scenario_h3_h10_M2_vs_M1_rmse_improvement_pct': 'M2 over M1, h3-h10', 'training_time_total_minutes': 'Training time (min)', 'methodological_role': 'Methodological role', 'notes': 'Notes'},
    'formats': {'Completed configs': 'count', 'Failed configs': 'count', 'RMSE M1': 'error_units', 'RMSE M2': 'error_units', 'RMSE M3': 'error_units', 'RMSE M4': 'error_units', 'MAE best operational': 'error_units', 'WAPE best operational': 'proportion_pct', 'M2 over M1 RMSE improvement': 'pct', 'M4 over M2 RMSE improvement': 'pct', 'M2 over M1, h0-h2': 'pct', 'M2 over M1, h3-h10': 'pct', 'Training time (min)': 'minutes'},
    'caption': 'Appendix Table ML-A0. Full model-family screening table',
    'note': 'Full SCREENING table including M4. This is not the final rolling-origin evaluation; it documents the model-family selection step.',
    'page_orientation': 'landscape',
    'own_page': True,
    'return_to_portrait': True,
}
render_table_from_source('Appendix ML-A0 - model-family screening full', 't04_model_family', full_config)


In [ ]:
# Render Table 8: synthetic weather input.

config = {
    'keep': ['horizon', 'alpha', 'climatology_weight', 'horizon_tier', 'weather_source'],
    'rename': {'horizon': 'Horizon', 'alpha': 'Realised-anchor weight', 'climatology_weight': 'Climatology weight', 'horizon_tier': 'Horizon group', 'weather_source': 'Weather source'},
    'formats': {'Horizon': 'count', 'Realised-anchor weight': 'proportion_pct', 'Climatology weight': 'proportion_pct'},
    'caption': 'Table 8. Synthetic weather input by forecast horizon',
    'note': 'Final daily M1/M2/M3 specification. The conservative climatology-drift schedule supersedes the earlier baseline alpha schedule for the final M2 results.',
}
render_table_from_source('Table 8 - synthetic weather input', 't08_alpha_schedule', config)


In [ ]:

# Render Figure B: origin schedule panel.

item_id = 'Figure B - origin schedule panel'
if require_sources(item_id, ['fig_b_daily_schedule', 'fig_b_step5_schedule']):
    daily = DATA['fig_b_daily_schedule'].copy()
    step5 = DATA['fig_b_step5_schedule'].copy()
    daily['origin_date'] = pd.to_datetime(daily['origin_date'])
    step5['origin_date'] = pd.to_datetime(step5['origin_date'])
    plotted_rows = pd.concat([
        daily.assign(series='Daily origin'),
        step5.assign(series='Five-day step'),
    ], ignore_index=True)
    log_figure_source(
        item_id,
        ['fig_b_daily_schedule', 'fig_b_step5_schedule'],
        plotted_rows[
            ['series', 'origin_date', 'origin_weekday', 'run_mode', 'origin_schedule_mode', 'n_origins', 'n_horizons']
        ],
    )

    fig, ax = plt.subplots(figsize=figure_size(0.58))
    rows = {'Daily-origin evaluation': 1.15, 'Older five-day-step': 0.25}
    train_start = -16
    train_end = -0.15
    horizons = np.arange(0, 11)
    for label, y in rows.items():
        ax.add_patch(
            Rectangle((train_start, y - 0.11), train_end - train_start, 0.22, facecolor='#E6E6E6', edgecolor='none')
        )
        ax.text(
            train_start + 0.2,
            y,
            'expanding training window',
            va='center',
            ha='left',
            fontsize=8.5,
            color='#333333',
        )
        ax.axvline(0, ymin=(y - 0.18) / 1.55, ymax=(y + 0.18) / 1.55, color=MODEL_PALETTE['M1'], linewidth=1.2)
        ax.scatter([0], [y], s=48, color=MODEL_PALETTE['M1'], zorder=4)
        ax.text(0, y + 0.16, 'origin h=0', va='bottom', ha='center', fontsize=8.5)
        ax.scatter(horizons[:3], np.full(3, y), s=42, color=MODEL_PALETTE['M2'], zorder=3)
        ax.scatter(horizons[3:], np.full(8, y), s=42, color=MODEL_PALETTE['M3'], zorder=3)
        for h in horizons:
            ax.text(h, y - 0.18, f'h{h}', ha='center', va='top', fontsize=7.5)
        ax.hlines(y, 0, 10, color='0.55', linewidth=0.8, zorder=1)
        if label.startswith('Daily'):
            ax.text(10.8, y, 'origins every day', va='center', ha='left', fontsize=8.5)
        else:
            for x in [-15, -10, -5, 0]:
                ax.axvline(
                    x,
                    ymin=(y - 0.06) / 1.55,
                    ymax=(y + 0.06) / 1.55,
                    color=MODEL_PALETTE['M1'],
                    linewidth=0.9,
                    alpha=0.7,
                )
            ax.text(10.8, y, 'origins every 5 days', va='center', ha='left', fontsize=8.5)

    ax.text(1.0, 1.50, 'Actual MEPS\nh0-h2', ha='center', va='center', color=MODEL_PALETTE['M2'], fontsize=8)
    ax.text(6.5, 1.50, 'Synthetic emulator\nh3-h10', ha='center', va='center', color=MODEL_PALETTE['M3'], fontsize=8)
    ax.set_yticks(list(rows.values()))
    ax.set_yticklabels(list(rows.keys()))
    ax.set_xlim(train_start - 0.5, 13.0)
    ax.set_ylim(-0.15, 1.55)
    ax.set_xlabel('Days relative to forecast origin')
    ax.set_title('Rolling-origin evaluation design', fontsize=OUTPUT_STYLE['figure_title_font_size_pt'])
    ax.spines['left'].set_visible(False)
    style_figure(ax, grid_axis='x')
    ax.tick_params(axis='y', length=0)
    ax.set_xticks([-15, -10, -5, 0, 5, 10])
    save_figure(
        item_id,
        'figure_B_origin_schedule_panel',
        fig,
        note='Pedagogical method schematic; schedule source files are printed above for audit.',
    )


## Results tables and figures

This section renders the regression and machine-learning result tables and figures from existing analysis outputs.


In [ ]:

# Render Table 5 and appendix regression coefficient tables.

REGRESSION_CURRENT_OBS_SIGNATURE = '46,841 (46)'
REGRESSION_KNOWN_STALE_OBS_SIGNATURE = '46,800 (46)'
REGRESSION_KNOWN_STALE_CONSTANT_PREFIX = '6.2696'


def _label_column(df):
    return df.columns[0]


def _lookup_stat(stats_df, label_col, stat_label, model_col):
    rows = stats_df.loc[stats_df[label_col].astype(str).eq(stat_label), model_col]
    return rows.iloc[0] if not rows.empty else np.nan


def _lookup_stat_kind(stats_df, label_col, stat_kind, model_col):
    labels = stats_df[label_col].map(lambda value: '' if pd.isna(value) else str(value))
    if stat_kind == 'observations':
        mask = labels.eq('Observations (#Stores)')
    elif stat_kind == 'aic_bic':
        mask = labels.eq('AIC/BIC')
    elif stat_kind == 'mape':
        mask = labels.eq('MAPE')
    elif stat_kind == 'adjusted_r2':
        # Avoid regex because Arrow-backed strings treat '?' as a repetition operator.
        mask = labels.map(lambda text: text.startswith('Adjusted R'))
    elif stat_kind == 'delta_adjusted_r2':
        mask = labels.map(lambda text: ('Adjusted R' in text) and ('(pp)' in text))
    elif stat_kind == 'f_weather':
        mask = labels.eq('F-test: weather block')
    elif stat_kind == 'f_added':
        mask = labels.eq('F-test: model-specific added weather terms')
    else:
        raise KeyError(f'Unknown regression statistic kind: {stat_kind}')
    rows = stats_df.loc[mask, model_col]
    return rows.iloc[0] if not rows.empty else np.nan


def _adjusted_r2_display(value):
    num = to_number(value)
    return '' if num is None else f'{num / 100.0:.3f}'


def _regression_ladder_current(panel_a, panel_b):
    a_label = _label_column(panel_a)
    b_label = _label_column(panel_b)
    model_cols = [col for col in panel_a.columns if col != a_label]
    first_model = model_cols[0]
    constant = str(_lookup_stat(panel_a, a_label, 'Constant', first_model))
    obs = str(_lookup_stat_kind(panel_b, b_label, 'observations', first_model))
    if REGRESSION_KNOWN_STALE_OBS_SIGNATURE in obs or constant.startswith(REGRESSION_KNOWN_STALE_CONSTANT_PREFIX):
        return False, f'source still matches old CampaignShare run: constant={constant}, observations={obs}'
    if REGRESSION_CURRENT_OBS_SIGNATURE not in obs:
        return False, f'cannot verify post-CampaignDay source signature: constant={constant}, observations={obs}'
    return True, f'post-CampaignDay signature detected: constant={constant}, observations={obs}'


def _split_regression_cell(value):
    if value is None or pd.isna(value):
        return None
    text = str(value).strip()
    if not text:
        return None
    match = re.match(r'^([-+]?\d+(?:\.\d+)?)(\*{1,3})?\s*\(([-+]?\d+(?:\.\d+)?)\)$', text)
    if match:
        coef, stars, t_stat = match.groups()
        return {'Coefficient': coef, 't-stat': t_stat, 'Stars': stars or ''}
    return {'Coefficient': text, 't-stat': '', 'Stars': ''}


def _coefficient_table(panel_a, model_col):
    label_col = _label_column(panel_a)
    rows = []
    for _, row in panel_a.iterrows():
        parsed = _split_regression_cell(row.get(model_col))
        if parsed is None:
            continue
        term = str(row[label_col]).replace('\n', ' - ')
        rows.append({'Term': term, **parsed})
    return pd.DataFrame(rows)


item_id = 'Table 5 - regression bottom statistics'
if require_sources(item_id, ['t05_regression_panel_a', 't05_regression_panel_b']):
    panel_a = DATA['t05_regression_panel_a'].copy()
    panel_b = DATA['t05_regression_panel_b'].copy()
    is_current, freshness_note = _regression_ladder_current(panel_a, panel_b)
    print('REGRESSION SOURCE CHECK - Table 5:', freshness_note)
    if not is_current:
        skip_item('Table 5 - regression bottom statistics', freshness_note + '; rerun notebook 03 and refresh TableX_booktabs1_panelA/B before rendering.')
        for suffix in ['i', 'ii', 'iii', 'iv', 'v']:
            skip_item(f'Appendix Table R1{suffix} - regression coefficients', freshness_note + '; coefficient appendix not rendered from stale source.')
    else:
        label_col = _label_column(panel_b)
        model_cols = [col for col in panel_b.columns if col != label_col]
        stat_specs = [
            ('Observations (#stores)', 'observations', 'raw'),
            ('AIC/BIC', 'aic_bic', 'raw'),
            ('MAPE', 'mape', 'raw'),
            ('Adjusted R\u00b2', 'adjusted_r2', 'adjusted_r2'),
            ('\u0394 Adjusted R\u00b2 (pp)', 'delta_adjusted_r2', 'raw'),
            ('F-test weather block', 'f_weather', 'raw'),
            ('F-test added terms', 'f_added', 'raw'),
        ]
        bottom_rows = []
        for display_label, stat_kind, formatter in stat_specs:
            row = {'Statistic': display_label}
            for model_col in model_cols:
                value = _lookup_stat_kind(panel_b, label_col, stat_kind, model_col)
                row[model_col] = _adjusted_r2_display(value) if formatter == 'adjusted_r2' else value
            bottom_rows.append(row)

        render_table_df(
            'Table 5 - regression bottom statistics',
            pd.DataFrame(bottom_rows),
            {
                'keep': ['Statistic', *model_cols],
                'default_format': 'raw',
                'caption': 'Table 5. Regression ladder bottom statistics',
                'note': 'Post-CampaignDay regression specification. Coefficients are reported separately in Appendix Tables R1a-R1e.',
                'narrowed': True,
            },
        )

        def _combined_coefficient_table(panel_a, combined_model_cols):
            label_col = _label_column(panel_a)
            rows = []
            for _, row in panel_a.iterrows():
                values = [row.get(model_col) for model_col in combined_model_cols]
                if all(pd.isna(value) or str(value).strip() == '' for value in values):
                    continue
                out_row = {'Term': str(row[label_col]).replace('\n', ' - ')}
                for model_col, value in zip(combined_model_cols, values):
                    out_row[model_col] = '' if pd.isna(value) else str(value).strip()
                rows.append(out_row)
            return pd.DataFrame(rows)

        combined_appendix_specs = [
            (
                'Appendix coefficients - models i-iii',
                model_cols[:3],
                'Appendix Table R1a. Regression coefficients for models (i)-(iii)',
                'Combined compact coefficient table for the control, linear weather, and linear-plus-quadratic weather specifications. Blank cells indicate terms not included in that model.',
            ),
            (
                'Appendix coefficients - models iv-v',
                model_cols[3:5],
                'Appendix Table R1b. Regression coefficients for models (iv)-(v)',
                'Combined compact coefficient table for the season-interacted linear and season-interacted linear-plus-quadratic weather specifications. Blank cells indicate terms not included in that model.',
            ),
        ]
        for appendix_item_id, appendix_model_cols, caption, note in combined_appendix_specs:
            coef_df = _combined_coefficient_table(panel_a, appendix_model_cols)
            if coef_df.empty:
                skip_item(appendix_item_id, f'no coefficients found for {caption}')
                continue
            render_table_df(
                appendix_item_id,
                coef_df,
                {
                    'keep': ['Term', *appendix_model_cols],
                    'default_format': 'raw',
                    'caption': caption,
                    'note': note,
                    'table_role': 'appendix',
                    'narrowed': True,
                },
            )


In [ ]:
# Render Table 6: category weather effects.

config = {
    'keep': ['Category', 'Ref. season', 'Adj. R² ctrl', 'Adj. R² full', 'Δ Adj. R²', 'W(all)', 'Sig. weather vars'],
    'rename': {'Ref. season': 'Reference season', 'Adj. R² ctrl': 'Adjusted R2 control', 'Adj. R² full': 'Adjusted R2 full', 'Δ Adj. R²': 'Delta adjusted R2 (pp)', 'W(all)': 'Weather block'},
    'formats': {'Adjusted R2 control': 'r2', 'Adjusted R2 full': 'r2', 'Delta adjusted R2 (pp)': 'pp'},
    'caption': 'Table 6. Category-level summary of weather effects',
    'note': 'Final category-level regression summary from notebook 03. Adjusted R-squared is shown as a decimal; delta is in percentage points.',
}
render_table_from_source('Table 6 - category weather effects', 't06_category_weather', config)


In [ ]:
# Render Table 7: basket focus pairs.

config = {
    'keep': ['A', 'B', 'co_orders', 'p_ab', 'lift', 'p_b_given_a', 'p_a_given_b', 'support_A', 'support_B'],
    'rename': {'A': 'Category A', 'B': 'Category B', 'co_orders': 'Co-orders', 'p_ab': 'Joint order share', 'lift': 'Lift', 'p_b_given_a': 'P(B | A)', 'p_a_given_b': 'P(A | B)', 'support_A': 'Support A', 'support_B': 'Support B'},
    'formats': {'Co-orders': 'count', 'Joint order share': 'proportion_pct', 'Lift': 'number_2', 'P(B | A)': 'proportion_pct', 'P(A | B)': 'proportion_pct', 'Support A': 'proportion_pct', 'Support B': 'proportion_pct'},
    'caption': 'Table 7. Basket-analysis focus pairs',
    'note': 'Focus pairs from notebook 02. Shares and conditional probabilities are proportions shown as percentages; lift is unitless.',
}
render_table_from_source('Table 7 - basket focus pairs', 't07_basket_focus', config)


In [ ]:

# Render Table 9: overall ML performance and improvements.

item_id = 'Table 9 - overall ML performance'
if require_sources(item_id, ['t09_ml_overall_hgroup']):
    source_df = DATA['t09_ml_overall_hgroup'].copy()
    overall = source_df.loc[source_df['horizon_group'].eq('h0_10')].copy()
    required_models = {'M1', 'M2', 'M3'}
    if not required_models.issubset(set(overall['model'])):
        skip_item(item_id, 'h0_10 rows for M1, M2, and M3 were not all present')
    else:
        by_model = overall.set_index('model')

        def _improvement_text(model):
            if model == 'M1':
                return 'Reference'
            if model == 'M2':
                rmse_imp = 100 * (by_model.loc['M1', 'rmse'] - by_model.loc['M2', 'rmse']) / by_model.loc['M1', 'rmse']
                mae_imp = 100 * (by_model.loc['M1', 'mae'] - by_model.loc['M2', 'mae']) / by_model.loc['M1', 'mae']
                return f'RMSE {rmse_imp:.1f}%; MAE {mae_imp:.1f}% vs M1'
            rmse_imp = 100 * (by_model.loc['M2', 'rmse'] - by_model.loc['M3', 'rmse']) / by_model.loc['M2', 'rmse']
            mae_imp = 100 * (by_model.loc['M2', 'mae'] - by_model.loc['M3', 'mae']) / by_model.loc['M2', 'mae']
            return f'RMSE {rmse_imp:.1f}%; MAE {mae_imp:.1f}% vs M2'

        table = overall.loc[overall['model'].isin(['M1', 'M2', 'M3']), ['model', 'rmse', 'mae', 'wape']].copy()
        table['Improvement'] = table['model'].map(_improvement_text)
        config = {
            'keep': ['model', 'rmse', 'mae', 'wape', 'Improvement'],
            'rename': {'model': 'Model', 'rmse': 'RMSE', 'mae': 'MAE', 'wape': 'WAPE'},
            'formats': {'RMSE': 'error_units', 'MAE': 'error_units', 'WAPE': 'proportion_pct', 'Improvement': 'raw'},
            'caption': 'Table 9. Overall store-category-day forecasting performance, h = 0 to h = 10',
            'note': 'Final daily rolling-origin evaluation. Metrics are pooled from row-level out-of-sample predictions. M1 is the no-weather baseline, M2 is the operational weather model, and M3 is the realised-weather benchmark.',
            'narrowed': True,
        }
        render_table_df(item_id, table, config)


In [ ]:

# Render Table 10: unit error reduction.

item_id = 'Table 10 - unit error reduction by aggregation level'
if require_sources(item_id, ['t10_store_category_units', 't10_store_day_units']):
    rows = []
    sc = DATA['t10_store_category_units']
    sd = DATA['t10_store_day_units']
    sc_row = sc.loc[sc['model'].eq('M2') & sc['horizon_group'].eq('h0_10')].head(1)
    sd_row = sd.loc[sd['model'].eq('M2') & sd['horizon_group'].eq('h0_10')].head(1)

    def _reduction_text(units, pct):
        if pd.isna(units) or pd.isna(pct):
            return ''
        return f'{units:.2f} ({pct:.1f}%)'

    if not sc_row.empty:
        r = sc_row.iloc[0]
        rows.append({
            'Aggregation level': 'Store-category-day',
            'Horizon group': 'h = 0 to h = 10',
            'Mean actual units': r['mean_y_true'],
            'MAE reduction': _reduction_text(r['mae_reduction_units_vs_M1'], r['mae_reduction_pct_vs_M1']),
            'RMSE reduction': _reduction_text(r['rmse_reduction_units_vs_M1'], r['rmse_reduction_pct_vs_M1']),
        })
    if not sd_row.empty:
        r = sd_row.iloc[0]
        rows.append({
            'Aggregation level': 'Store-day',
            'Horizon group': 'h = 0 to h = 10',
            'Mean actual units': r['mean_actual_store_day'],
            'MAE reduction': _reduction_text(r['mae_reduction_units_vs_M1'], r['mae_reduction_pct_vs_M1']),
            'RMSE reduction': _reduction_text(r['rmse_reduction_units_vs_M1'], r['rmse_reduction_pct_vs_M1']),
        })
    if not rows:
        skip_item(item_id, 'no M2 h0_10 rows found in unit-reduction sources')
    else:
        config = {
            'keep': ['Aggregation level', 'Horizon group', 'Mean actual units', 'MAE reduction', 'RMSE reduction'],
            'formats': {'Mean actual units': 'error_units', 'MAE reduction': 'raw', 'RMSE reduction': 'raw'},
            'caption': 'Table 10. Forecast-error reduction in units by aggregation level',
            'note': 'Final daily M2 versus M1 comparison. Reductions are lower forecast error, not additional sales or direct economic savings.',
            'narrowed': True,
        }
        render_table_df(item_id, pd.DataFrame(rows), config)


In [ ]:

# Render Table 11: category M2 improvement.

item_id = 'Table 11 - category M2 over M1 improvement'
if require_sources(item_id, ['t11_category_improvement']):
    source_df = DATA['t11_category_improvement'].copy()
    category = source_df.loc[source_df['horizon_group'].eq('h3_10')].copy()
    if category.empty:
        skip_item(item_id, 'no h3_10 category rows found')
    else:
        def _reduction_text(units, pct):
            if pd.isna(units) or pd.isna(pct):
                return ''
            return f'{units:.2f} ({pct:.1f}%)'

        category['RMSE reduction'] = category.apply(lambda r: _reduction_text(r['rmse_reduction_units'], r['rmse_reduction_pct']), axis=1)
        category['MAE reduction'] = category.apply(lambda r: _reduction_text(r['mae_reduction_units'], r['mae_reduction_pct']), axis=1)
        category = category.sort_values('rmse_reduction_pct', ascending=False).reset_index(drop=True)
        config = {
            'keep': ['Analyse_Kategori', 'mean_actual_units', 'RMSE reduction', 'MAE reduction'],
            'rename': {'Analyse_Kategori': 'Category', 'mean_actual_units': 'Mean actual units'},
            'formats': {'Mean actual units': 'error_units', 'RMSE reduction': 'raw', 'MAE reduction': 'raw'},
            'caption': 'Table 11. M2 improvement over M1 by category, h = 3 to h = 10',
            'note': 'Final daily category comparison for the medium-range emulator window. Reduction cells show units and percentage reduction in parentheses.',
            'narrowed': True,
        }
        render_table_df(item_id, category, config)


In [ ]:
# Render Figure C: RMSE by horizon.

item_id = 'Figure C - RMSE by horizon'
if require_sources(item_id, ['fig_c_metrics_by_horizon']):
    df = DATA['fig_c_metrics_by_horizon'].copy()
    df = df.loc[df['feature_set'].isin(['M1', 'M2', 'M3'])]
    log_figure_source(item_id, ['fig_c_metrics_by_horizon'], df[['feature_set', 'horizon', 'rmse']])
    fig, ax = plt.subplots(figsize=figure_size())
    for model, model_df in df.groupby('feature_set', sort=True):
        ax.plot(
            model_df['horizon'],
            model_df['rmse'],
            marker='o',
            linewidth=2,
            color=MODEL_PALETTE.get(model, SINGLE_SERIES_COLOR),
            label=model,
        )
    ax.set_xlabel('Forecast horizon (days)')
    ax.set_ylabel('RMSE (units)')
    ax.set_title('RMSE by forecast horizon')
    style_figure(ax, grid_axis='y')
    ax.legend(frameon=False)
    save_figure(item_id, 'figure_C_rmse_by_horizon', fig)


In [ ]:
# Render Figure D: M2 percentage improvement.

item_id = 'Figure D - M2 percentage improvement by horizon'
if require_sources(item_id, ['fig_d_m2_improvement']):
    df = DATA['fig_d_m2_improvement'].copy()
    log_figure_source(item_id, ['fig_d_m2_improvement'], df[['horizon', 'rmse_impr_pct_M2_vs_M1']])
    fig, ax = plt.subplots(figsize=figure_size())
    ax.plot(df['horizon'], df['rmse_impr_pct_M2_vs_M1'], marker='o', linewidth=2, color=SINGLE_SERIES_COLOR)
    ax.set_xlabel('Forecast horizon (days)')
    ax.set_ylabel('RMSE reduction vs M1 (%)')
    ax.set_title('M2 improvement over M1 by horizon')
    style_figure(ax, grid_axis='y')
    save_figure(item_id, 'figure_D_m2_pct_improvement_by_horizon', fig)


In [ ]:
# Render Figure E: store-day MAE.

item_id = 'Figure E - store-day MAE by horizon'
if require_sources(item_id, ['fig_e_store_day_horizon']):
    df = DATA['fig_e_store_day_horizon'].copy()
    df = df.loc[df['feature_set'].isin(['M1', 'M2', 'M3'])]
    log_figure_source(item_id, ['fig_e_store_day_horizon'], df[['feature_set', 'horizon', 'mae']])
    fig, ax = plt.subplots(figsize=figure_size())
    for model, model_df in df.groupby('feature_set', sort=True):
        ax.plot(
            model_df['horizon'],
            model_df['mae'],
            marker='o',
            linewidth=2,
            color=MODEL_PALETTE.get(model, SINGLE_SERIES_COLOR),
            label=model,
        )
    ax.set_xlabel('Forecast horizon (days)')
    ax.set_ylabel('Store-day MAE (units)')
    ax.set_title('Store-day MAE by horizon')
    style_figure(ax, grid_axis='y')
    ax.legend(frameon=False)
    save_figure(item_id, 'figure_E_store_day_mae_by_horizon', fig)


In [ ]:
# Render Figure F: cumulative h1-h8 cycle error.
item_id = 'Figure F - cumulative h1-h8 forecast error'
if require_sources(item_id, ['fig_f_replenishment_cycle']):
    order = ['M1', 'M2', 'M3']
    df = DATA['fig_f_replenishment_cycle'].set_index('model').loc[order]   # regenerated 26.06; cols: model, rmse, mae
    log_figure_source(item_id, ['fig_f_replenishment_cycle'], df.reset_index()[['model', 'rmse', 'mae']])
    fig, ax = plt.subplots(figsize=figure_size(0.62))
    bars = ax.bar(order, df['rmse'], width=0.62, color=[MODEL_PALETTE[m] for m in order])
    ax.set_xlabel('Model')
    ax.set_ylabel('Cumulative h1-8 cycle RMSE (units)')
    ax.set_ylim(0, df['rmse'].max() * 1.15)
    ax.bar_label(bars, fmt='%.0f', padding=3, fontsize=10, color='0.20')
    style_figure(ax, grid_axis='y')
    save_figure(item_id, 'figure_F_cumulative_h1_h8_error', fig)


In [ ]:
# Render Figure G: category horizon heatmap.

item_id = 'Figure G - category horizon heatmap'
if require_sources(item_id, ['fig_g_category_horizon']):
    df = DATA['fig_g_category_horizon'].copy()
    log_figure_source(item_id, ['fig_g_category_horizon'], df[['Analyse_Kategori', 'horizon', 'rmse_impr_pct_M2_vs_M1']])
    heat = df.pivot(index='Analyse_Kategori', columns='horizon', values='rmse_impr_pct_M2_vs_M1')
    heat = heat.rename(index=TRANSLATION)
    fig, ax = plt.subplots(figsize=figure_size(0.72))
    im = ax.imshow(heat.values, aspect='auto', cmap=HEATMAP_CMAP)
    ax.set_xticks(range(len(heat.columns)))
    ax.set_xticklabels(heat.columns)
    ax.set_yticks(range(len(heat.index)))
    ax.set_yticklabels(heat.index)
    ax.set_xlabel('Forecast horizon')
    ax.set_ylabel('Category')
    ax.set_title('M2 RMSE improvement by category and horizon')
    for row_index in range(heat.shape[0]):
        for col_index in range(heat.shape[1]):
            value = heat.iloc[row_index, col_index]
            text_color = 'white' if pd.notna(value) and value > np.nanmax(heat.values) * 0.58 else '#1F1F1F'
            ax.text(
                col_index,
                row_index,
                f'{value:.1f}%' if pd.notna(value) else '-',
                ha='center',
                va='center',
                color=text_color,
                fontsize=8,
            )
    cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.025)
    cbar.set_label('RMSE improvement %, M2 vs M1')
    style_figure(ax, grid_axis=None)
    save_figure(item_id, 'figure_G_category_horizon_heatmap', fig)


In [ ]:
# Render Figure H: unusual weather days.

item_id = 'Figure H - unusual weather days'
if require_sources(item_id, ['fig_h_weather_tail']):
    df = DATA['fig_h_weather_tail'].copy()
    df['tail_group_label'] = df['tail_group'].map(TRANSLATION).fillna(df['tail_group'])
    log_figure_source(item_id, ['fig_h_weather_tail'], df[['tail_group', 'mean_actual_units', 'rmse_M1', 'rmse_M2', 'rmse_impr_pct_M2_vs_M1']])
    fig, ax = plt.subplots(figsize=figure_size(0.55))
    ax.bar(df['tail_group_label'], df['rmse_impr_pct_M2_vs_M1'], color=SINGLE_SERIES_COLOR)
    ax.set_ylabel('RMSE reduction vs M1 (%)')
    ax.set_title('M2 improvement on unusual weather days')
    ax.tick_params(axis='x', rotation=25)
    style_figure(ax, grid_axis='y')
    save_figure(item_id, 'figure_H_unusual_weather_days', fig)


In [ ]:
# Render Figure I: seasonal forecast improvement.

item_id = 'Figure I - forecast improvement by season'
if require_sources(item_id, ['fig_i_season']):
    df = DATA['fig_i_season'].copy()
    df['season_label'] = df['season'].map(TRANSLATION).fillna(df['season'])
    log_figure_source(item_id, ['fig_i_season'], df[['horizon_group', 'season', 'rmse_M1', 'rmse_M2', 'rmse_impr_pct_M2_vs_M1']])
    fig, ax = plt.subplots(figsize=figure_size(0.58))
    group_colors = {'h0_10': MODEL_PALETTE['M2'], 'h0_2': MODEL_PALETTE['M1'], 'h3_10': MODEL_PALETTE['M3']}
    for group, group_df in df.groupby('horizon_group', sort=True):
        ax.plot(
            group_df['season_label'],
            group_df['rmse_impr_pct_M2_vs_M1'],
            marker='o',
            linewidth=2,
            color=group_colors.get(group, SINGLE_SERIES_COLOR),
            label=TRANSLATION.get(group, group),
        )
    ax.set_ylabel('RMSE reduction vs M1 (%)')
    ax.set_title('Forecast improvement by season')
    style_figure(ax, grid_axis='y')
    ax.legend(frameon=False)
    save_figure(item_id, 'figure_I_season_improvement', fig)


In [ ]:
# Render Figure J: grouped feature importance.

item_id = 'Figure J - grouped M2 feature importance'
if require_sources(item_id, ['fig_j_importance']):
    df = DATA['fig_j_importance'].copy()
    value_col = 'mean_abs_shap' if 'mean_abs_shap' in df.columns else df.select_dtypes(include='number').columns[-1]
    df = df.sort_values(value_col, ascending=True)
    log_figure_source(item_id, ['fig_j_importance'], df[['feature_group', value_col]])
    fig, ax = plt.subplots(figsize=figure_size(0.72))
    ax.barh(df['feature_group'], df[value_col], color=SINGLE_SERIES_COLOR)
    ax.set_xlabel('Mean absolute SHAP value')
    ax.set_title('Grouped feature importance for M2')
    style_figure(ax, grid_axis='x')
    save_figure(item_id, 'figure_J_grouped_feature_importance_m2', fig, note='Exploratory importance source; not from the final-daily fits.')
    print('Note: grouped importance is exploratory and is not from the final-daily fits.')


In [ ]:

# Render Table 13: weather-variable SHAP importance.

item_id = 'Table 13 - weather-variable SHAP importance'
if require_sources(item_id, ['t18_weather_shap']):
    shap_weather = DATA['t18_weather_shap'].copy()
    shap_weather['weather_variable'] = shap_weather['weather_variable'].map({
        'temp': 'Temperature',
        'temperature': 'Temperature',
        'cloud': 'Cloud cover',
        'precip': 'Precipitation',
        'precipitation': 'Precipitation',
        'humid': 'Humidity',
        'humidity': 'Humidity',
        'wind': 'Wind speed',
    }).fillna(shap_weather['weather_variable'])
    shap_weather = shap_weather.sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
    config = {
        'keep': ['weather_variable', 'mean_abs_shap', 'mean_abs_shap_share', 'n_models'],
        'rename': {'weather_variable': 'Weather variable', 'mean_abs_shap': 'Mean |SHAP|', 'mean_abs_shap_share': 'Share of |SHAP|', 'n_models': 'Models'},
        'formats': {'Mean |SHAP|': 'number_2', 'Share of |SHAP|': 'proportion_pct', 'Models': 'count'},
        'caption': 'Table 13. Weather-variable SHAP importance for M2',
        'note': 'Exploratory SHAP diagnostics from full_step5 fitted models, not the final daily run. Values describe model usage, not causal effects.',
        'narrowed': True,
    }
    render_table_df(item_id, shap_weather, config)


In [ ]:
# Render Figure K: daily-origin versus five-day-step comparison.

item_id = 'Figure K - daily-origin versus five-day-step'
if require_sources(item_id, ['fig_k_daily_vs_step5']):
    df = DATA['fig_k_daily_vs_step5'].copy()
    log_figure_source(item_id, ['fig_k_daily_vs_step5'], df[['horizon', 'rmse_m2_daily_origin', 'rmse_m2_full_step5', 'rmse_diff_pct_daily_vs_step5']])
    fig, ax = plt.subplots(figsize=figure_size())
    ax.plot(df['horizon'], df['rmse_m2_daily_origin'], marker='o', linewidth=2, color=MODEL_PALETTE['M2'], label='Daily origin')
    ax.plot(df['horizon'], df['rmse_m2_full_step5'], marker='s', linewidth=2, color=MODEL_PALETTE['M1'], label='Five-day step')
    ax.set_xlabel('Forecast horizon (days)')
    ax.set_ylabel('M2 RMSE (units)')
    ax.set_title('Daily-origin versus five-day-step evaluation')
    style_figure(ax, grid_axis='y')
    ax.legend(frameon=False)
    save_figure(item_id, 'figure_K_daily_origin_vs_step5', fig)


## Appendix tables and figures

This section renders supplementary regression, robustness and exploratory machine-learning tables for the thesis appendix.


In [ ]:
# Render Appendix ML-A1: full-step M1-M4 comparison.

config = {
    'keep': ['horizon', 'horizon_tier', 'M1_RMSE', 'M2_RMSE', 'M3_RMSE', 'M4_RMSE', 'M1_MAE', 'M2_MAE', 'M3_MAE', 'M4_MAE', 'M2_vs_M1_RMSE_improvement_pct', 'M4_vs_M2_RMSE_improvement', 'n'],
    'rename': {'horizon': 'Horizon', 'horizon_tier': 'Horizon group', 'M1_RMSE': 'RMSE M1', 'M2_RMSE': 'RMSE M2', 'M3_RMSE': 'RMSE M3', 'M4_RMSE': 'RMSE M4', 'M1_MAE': 'MAE M1', 'M2_MAE': 'MAE M2', 'M3_MAE': 'MAE M3', 'M4_MAE': 'MAE M4', 'M2_vs_M1_RMSE_improvement_pct': 'M2 over M1 RMSE improvement', 'M4_vs_M2_RMSE_improvement': 'M4 over M2 RMSE reduction', 'n': 'Rows'},
    'formats': {'Horizon': 'count', 'RMSE M1': 'error_units', 'RMSE M2': 'error_units', 'RMSE M3': 'error_units', 'RMSE M4': 'error_units', 'MAE M1': 'error_units', 'MAE M2': 'error_units', 'MAE M3': 'error_units', 'MAE M4': 'error_units', 'M2 over M1 RMSE improvement': 'pct', 'M4 over M2 RMSE reduction': 'error_units', 'Rows': 'count'},
    'caption': 'Appendix Table ML-A1. Initial full-step M1/M2/M3/M4 comparison',
    'note': 'Superseded five-day-step run. M4 is exploratory, built on the synthetic emulator, and is not carried into the final analysis.',
    'page_orientation': 'landscape',
    'own_page': True,
    'return_to_portrait': True,
}
render_table_from_source('Appendix ML-A1 - full-step M1-M4', 't12_full_step5', config)


In [ ]:

# Render Table 12: regional performance.

config = {
    'keep': ['Region', 'n_stores', 'rmse_M1', 'rmse_M2', 'rmse_impr_pct_M2_vs_M1'],
    'rename': {'n_stores': 'Stores', 'rmse_M1': 'RMSE M1', 'rmse_M2': 'RMSE M2', 'rmse_impr_pct_M2_vs_M1': 'RMSE reduction (%)'},
    'formats': {'Stores': 'count', 'RMSE M1': 'error_units', 'RMSE M2': 'error_units', 'RMSE reduction (%)': 'pct'},
    'fill_values': {'RMSE reduction (%)': '-'},
    'caption': 'Table 12. Regional M2 performance relative to M1',
    'note': 'Final daily regional diagnostic. Small regional samples should be interpreted descriptively, not as a separate model-selection result.',
    'narrowed': True,
}
render_table_from_source('Table 12 - regional M2 performance', 't13_region_summary', config)


In [ ]:
# Render Appendix ML-A3: alpha sensitivity.

config = {
    'keep': ['scenario', 'rmse_M1', 'rmse_M2', 'RMSE_improvement_pct_vs_M1', 'MAE_improvement_pct_vs_M1'],
    'rename': {'scenario': 'Scenario', 'rmse_M1': 'RMSE M1', 'rmse_M2': 'RMSE M2', 'RMSE_improvement_pct_vs_M1': 'RMSE reduction (%)', 'MAE_improvement_pct_vs_M1': 'MAE reduction (%)'},
    'formats': {'RMSE M1': 'error_units', 'RMSE M2': 'error_units', 'RMSE reduction (%)': 'pct', 'MAE reduction (%)': 'pct'},
    'fill_values': {'RMSE reduction (%)': '-', 'MAE reduction (%)': '-'},
    'caption': 'Appendix Table ML-A3. Alpha-sensitivity of the medium-range emulator',
    'note': 'Robustness table for the medium-range emulator. This is an appendix sensitivity result, not the main final-daily specification table.',
    'narrowed': True,
}
render_table_from_source('Appendix ML-A3 - alpha sensitivity', 't14_alpha_sensitivity', config)


In [ ]:
# Render Appendix A.1: VIF diagnostics.

panel_a_config = {
    'keep': ['Model', 'Mean VIF', 'Max VIF', 'Variable with max VIF'],
    'formats': {'Mean VIF': 'number_2', 'Max VIF': 'number_2'},
    'caption': 'Appendix Table A.1a. VIF summary by model',
    'note': 'VIF diagnostics from notebook 03. Values are variance inflation factors.',
}
render_table_from_source('Appendix A.1a - VIF summary', 't15_vif_panel_a', panel_a_config)

panel_b_config = {
    'keep': ['Statistic', 'Value'],
    'formats': {'Value': 'number_2'},
    'caption': 'Appendix Table A.1b. Key weather-term VIF values',
    'note': 'Selected VIF diagnostics for weather terms from notebook 03.',
}
render_table_from_source('Appendix A.1b - VIF detail', 't15_vif_panel_b', panel_b_config)


In [ ]:
# Render Appendix A.2: significance diagnostics.

panel_a_config = {
    'keep': ['Season', 'Weather var', 'Linear sig. (%)', 'Quadratic sig. (%)'],
    'rename': {'Weather var': 'Weather variable'},
    'formats': {'Linear sig. (%)': 'pct', 'Quadratic sig. (%)': 'pct'},
    'caption': 'Appendix Table A.2a. Significant weather terms by season',
    'note': 'Shares are percentages of category-season models with significant weather terms.',
}
render_table_from_source('Appendix A.2a - seasonal significance', 't16_significance_panel_a', panel_a_config)

panel_b_config = {
    'keep': ['Weather var', 'Linear sig. (%)', 'Quadratic sig. (%)'],
    'rename': {'Weather var': 'Weather variable'},
    'formats': {'Linear sig. (%)': 'pct', 'Quadratic sig. (%)': 'pct'},
    'caption': 'Appendix Table A.2b. Significant weather terms overall',
    'note': 'Overall shares of significant weather terms across category-season models.',
}
render_table_from_source('Appendix A.2b - overall significance', 't16_significance_panel_b', panel_b_config)


In [ ]:
# Render Appendix A.3: IQR effects.

config = {
    'keep': ['Category', 'Season', 'AT IQR effect', 'Precip IQR effect', 'Cloud IQR effect'],
    'rename': {'AT IQR effect': 'Apparent temperature IQR effect', 'Precip IQR effect': 'Precipitation IQR effect', 'Cloud IQR effect': 'Cloud-cover IQR effect'},
    'formats': {'Apparent temperature IQR effect': 'regression', 'Precipitation IQR effect': 'regression', 'Cloud-cover IQR effect': 'regression'},
    'caption': 'Appendix Table A.3. Seasonal interquartile-range effects by category',
    'note': 'Regression-based IQR effects from notebook 03. Coefficients are rounded to three decimals with significance stars retained.',
}
render_table_from_source('Appendix A.3 - IQR effects', 't17_iqr_effects', config)


In [ ]:
# Register skipped Figure L.

skip_item('Figure L - quadratic realised-weather category curves', 'OUT OF SCOPE for notebook 12. This requires regression coefficients/model objects and must be generated in notebook 03 where the regression is fitted.')


In [ ]:
# Register skipped Figure M.

skip_item('Figure M - stylised weather-shift quantity-vs-price', 'OUT OF SCOPE for notebook 12. No existing source file was found; this must be generated in notebook 03 where the regression is fitted, rather than recomputed here.')


In [ ]:
# Save the combined Word document and print the manifest.
DOCX_WRITE_PATH = timestamp_variant(OUTPUT_DOCX_PATH)
DOC.save(DOCX_WRITE_PATH)
MANIFEST['docx'] = str(DOCX_WRITE_PATH)

print('Combined Word document written:')
print(' -', DOCX_WRITE_PATH.relative_to(PROJECT_ROOT))
print('\nTables written:')
for item in MANIFEST['tables']:
    print(f" - {item['item']}: {item['rows']} rows x {len(item['columns'])} columns")
print('\nNarrowed tables:')
if MANIFEST['narrowed_tables']:
    for item in MANIFEST['narrowed_tables']:
        print(f" - {item['item']}: {len(item['columns'])} columns")
else:
    print(' - None')
print('\nFigures written:')
for item in MANIFEST['figures']:
    print(
        f" - {item['item']}: {Path(item['pdf']).relative_to(PROJECT_ROOT)}; "
        f"{Path(item['png']).relative_to(PROJECT_ROOT)}; style={item.get('style', '')}"
    )
print('\nFigure sources confirmed:')
if MANIFEST['figure_sources']:
    for item in MANIFEST['figure_sources']:
        print(f" - {item['item']}: " + '; '.join(item['sources']))
else:
    print(' - None')
print(
    f'\nShared figure style applied: {FIGURE_STYLE_NAME}; '
    f'palette M1={MODEL_PALETTE["M1"]}, M2={MODEL_PALETTE["M2"]}, '
    f'M3={MODEL_PALETTE["M3"]}, M4={MODEL_PALETTE["M4"]}.'
)
print('\nPortrait A4 fit flags for Torkel discussion:')
if MANIFEST['fit_flags']:
    seen = set()
    for item in MANIFEST['fit_flags']:
        if item['item'] in seen:
            continue
        seen.add(item['item'])
        print(f" - {item['item']}: {item['columns']} columns; {item['reason']}")
else:
    print(' - None')

print('\nSkipped items:')
if MANIFEST['skipped']:
    for item in MANIFEST['skipped']:
        print(f" - {item['item']}: {item['reason']}")
else:
    print(' - None')
